In [ ]:
import numpy
import seaborn

from matplotlib import pyplot

from pristine import load_pristine_rv_curve, load_pristine_timeseries
from realistic import load_realistic_timeseries, load_realistic_timeseries_curve


In [ ]:
a_limit = 150
sample_size = 10


In [ ]:
rts = load_realistic_timeseries()
pts = load_pristine_timeseries()
rts.head()


In [ ]:
sample = rts[rts['a'] < a_limit]
if len(sample) > sample_size:
    sample = sample.sample(n=sample_size, random_state=1)

for _, ts in sample.iterrows():
    pristine = pts[pts['simulation_id'].astype(str) == str(ts['pristine_id'])].iloc[0]
    pristine_rv = load_pristine_rv_curve(pristine)
    realistic_curve = load_realistic_timeseries_curve(ts)
    period = ts['Porb']

    print(ts[['realistic_id', 'pristine_id', 'a', 'e', 'periapsis_phase', 'Macc', 'Mcomp', 'Rcomp', 'Porb', 'tau_eff', 'RV accuracy', 'cadence', 'baseline']])

    seaborn.lineplot(x=pristine_rv['rv_times'], y=pristine_rv['rv'], c='red')
    seaborn.scatterplot(x=realistic_curve['rv_times'], y=realistic_curve['rv'])
    pyplot.xlabel('JD')
    pyplot.ylabel('RV [m / s]')
    pyplot.show()
    pyplot.close()

    folded_pristine_times = (numpy.asarray(pristine_rv['rv_times']) - ts['peak_time']) % period
    folded_times = (numpy.asarray(realistic_curve['rv_times']) - ts['peak_time']) % period
    seaborn.lineplot(x=folded_pristine_times, y=pristine_rv['rv'], c='red')
    seaborn.scatterplot(x=folded_times, y=realistic_curve['rv'])
    pyplot.xlabel('Phase time [d]')
    pyplot.ylabel('RV [m / s]')
    pyplot.show()
    pyplot.close()
